© 2026 Mechanome Bio. Proprietary and confidential.

# Running Cellist
repo: https://github.com/wanglabtongji/Cellist , docs: https://cellist.readthedocs.io/en/latest/

- Python >=3.10. Now on PyPI (`cellist`, import name `Cellist`), so no clone needed. Installed in the shared env.
- No Xenium tutorial or Xenium code in the repo; follows the seqFISH+ tutorial (imaging platform, closest to Xenium).
- Steps: bin transcripts into 5-pixel bins (gem file + spot counts) -> nucleus files from the 10x nucleus
  boundaries (instead of Cellist's watershed) -> `cellist seg` (tutorial step 3).
- Cellist's convention: gem `x` = image row, `y` = image column. Followed throughout.

In [ ]:
import os
import sys

import numpy
import pandas
from skimage.measure import regionprops_table

sys.path.append("..")
from utils import seg_utils

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
METHOD = "cellist"
BOUNDING_BOX_MICRONS = (4673, 1476, 5036, 1805)  # microns; None = whole slide

BIN_SIZE = 5  # pixels, as in the seqFISH+ tutorial
PREFIX = DATASET_NAME
WORK_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "Cellist" / DATASET_NAME
RESULT_DIRECTORY = WORK_DIRECTORY / "result"
WORK_DIRECTORY.mkdir(parents=True, exist_ok=True)

sdata = seg_utils.load_sdata(DATASET_NAME)

In [ ]:
seg_utils.preview_bounding_box(sdata, BOUNDING_BOX_MICRONS)

In [ ]:
sdata_crop = seg_utils.crop_sdata(sdata, BOUNDING_BOX_MICRONS)
image_shape = seg_utils.get_image(sdata_crop).shape
origin_pixels = seg_utils.image_origin_pixels(sdata_crop)

## Gem file: 5-pixel bins
Columns `geneID, x, y, MIDCount`. Bin coordinates are the bin's top-left pixel (multiples of 5), as in the tutorial.

In [ ]:
transcripts = seg_utils.get_transcripts(sdata_crop)
transcripts = transcripts[(transcripts["x_pixels"] >= 0) & (transcripts["y_pixels"] >= 0)]
gem = pandas.DataFrame({
    "geneID": transcripts[seg_utils.GENE_COLUMN].astype(str).values,
    "x": (transcripts["y_pixels"] // BIN_SIZE * BIN_SIZE).astype(int).values,  # row
    "y": (transcripts["x_pixels"] // BIN_SIZE * BIN_SIZE).astype(int).values,  # column
    "MIDCount": 1,
})
gem = gem.groupby(["geneID", "x", "y"], as_index=False)["MIDCount"].sum()
gem_path = WORK_DIRECTORY / f"{PREFIX}_bin{BIN_SIZE}.txt"
gem.to_csv(gem_path, sep="\t", index=False)

## Nucleus files from the 10x nucleus boundaries
Mirrors what `cellist watershed` writes (`Cellist/Watershed.py`), with the rasterised 10x nuclei in place of the
watershed labels. Files are named "Watershed" because `cellist seg` only accepts Watershed or Cellpose.

In [ ]:
from Cellist.IO import gem_to_mat
from Cellist.Watershed import write_segmentation_cell, write_segmentation_coord

nucleus_mask = seg_utils.polygons_to_mask(sdata_crop[seg_utils.NUCLEI_KEY], origin_pixels, image_shape)

spot_counts_path = WORK_DIRECTORY / f"{PREFIX}_bin{BIN_SIZE}.h5"
gem_with_spot_ids = gem_to_mat(gem_df=gem.copy(), outfile=str(spot_counts_path), countname="MIDCount")
spot_coordinates = gem_with_spot_ids.drop_duplicates(subset=["x_y"])[["x", "y", "x_y"]]

nucleus_spots = write_segmentation_coord(nucleus_mask, spot_coordinates, str(WORK_DIRECTORY), PREFIX, seg_method="Watershed")
write_segmentation_cell(nucleus_spots, gem_with_spot_ids, PREFIX, str(WORK_DIRECTORY), seg_method="Watershed")
pandas.DataFrame(regionprops_table(nucleus_mask, properties=["label", "area", "centroid", "equivalent_diameter_area"])).to_csv(
    WORK_DIRECTORY / f"{PREFIX}_Watershed_nucleus_property.txt", sep="\t", index=False)

## Cell segmentation
Arguments from the seqFISH+ tutorial, except `--resolution` (microns per gem unit = Xenium pixel size).

In [ ]:
# built as a string and run with ! so the output shows in the notebook
cellist_command = " ".join([
    "cellist", "seg", "--platform", "imaging",
    "--resolution", str(seg_utils.PIXEL_SIZE_MICRONS),
    "--gem", str(gem_path),
    "--spot-count-h5", str(spot_counts_path),
    "--nucleus-seg-method", "Watershed",
    "--nucleus-prop", str(WORK_DIRECTORY / f"{PREFIX}_Watershed_nucleus_property.txt"),
    "--nucleus-count-h5", str(WORK_DIRECTORY / f"{PREFIX}_Watershed_segmentation_cell_count.h5"),
    "--nucleus-seg", str(WORK_DIRECTORY / f"{PREFIX}_Watershed_nucleus_coord.txt"),
    "--nworkers", str(os.cpu_count()),
    "--cell-radius", "15", #probably an important parameter to tune 
    "--spot-imputation-distance", "2.5", #also ^
    "--noise-prop", "0",
    "--outdir", str(RESULT_DIRECTORY),
    "--outprefix", PREFIX,
])
!{cellist_command}

## Save
`<prefix>_Cellist_segmentation.txt` assigns each bin to a cell. Painted back onto the pixel grid as a label mask, then polygons.

In [ ]:
assignment_path = sorted(RESULT_DIRECTORY.glob(f"*/{PREFIX}_Cellist_segmentation.txt"))[-1]
assignment = pandas.read_csv(assignment_path, sep="\t")
assignment["Cellist"] = assignment["Cellist"].fillna(0).astype(int)
assignment = assignment[assignment["Cellist"] > 0]

bin_mask = numpy.zeros((image_shape[0] // BIN_SIZE + 1, image_shape[1] // BIN_SIZE + 1), dtype=numpy.int32)
bin_mask[assignment["x"] // BIN_SIZE, assignment["y"] // BIN_SIZE] = assignment["Cellist"]
mask = numpy.kron(bin_mask, numpy.ones((BIN_SIZE, BIN_SIZE), dtype=numpy.int32))[:image_shape[0], :image_shape[1]]

polygons = seg_utils.mask_to_polygons(mask, origin_pixels=origin_pixels)
seg_utils.save_segmentation(sdata, polygons, METHOD, DATASET_NAME, BOUNDING_BOX_MICRONS)